# Symbolic student — hardware handoff example

Loads a trained arm, checks the numpy reference against the golden vectors, plots the physics,
and sweeps input precision to show where the fixed-point budget actually binds.

**Runs on numpy + matplotlib only.** No TensorFlow, no Keras, no qkeras — so any environment
works, including one you build for HLS. The last cell is optional and does need TF.

Model: `FORMULA.md` has the whole thing written out with the trained constants substituted.
Interfaces, ADC boundary and acceptance criteria: `README.md`.

In [ ]:
# ---- 1. load an arm -------------------------------------------------------
import os, sys, json
import numpy as np
import matplotlib.pyplot as plt

HERE = os.getcwd()
sys.path.insert(0, HERE)
from symbolic_ref import digitize, forward, decode

ARM = "digi_2bit_paper_code_nexp1"        # <- swap to ..._nexp4 for the MoE
EXP = os.path.join(HERE, "exports", ARM)

C = json.load(open(os.path.join(EXP, "constants.json")))
g = np.load(os.path.join(EXP, "golden.npz"))
ls = g["labels_scale"]

print(f"arm           : {C['arm']}")
print(f"experts       : {C['n_experts']}   total params: {C['total_params']}")
print(f"ADC           : {C['digitize']['n_bits']}-bit, thresholds {C['digitize']['thresholds_e']} e-")
print(f"                {C['digitize']['comparator']}")
print(f"labels_scale  : {np.round(ls, 4).tolist()}")
print(f"golden events : {len(g['y_pred_14'])}")
print("\nper-expert physics scalars (everything else is the 354-param uncertainty MLP):")
for k, v in C["experts"][0].items():
    if not k.startswith("mlp_"):
        print(f"  {k:<16} {round(v, 6) if isinstance(v, float) else np.round(v, 6)}")

## 2. Bit-match check

`y_pred_14` is what the trained Keras model emits. The numpy reference should reproduce it to
float32 rounding. **Swap `forward()` for your C++/HLS output here** and this becomes your
hardware acceptance test.

In [ ]:
# ---- 2. numpy reference vs golden ----------------------------------------
codes = digitize(g["charge_analog_e"], C)              # analog e- -> 2-bit codes
y_ref = forward(g["adc_codes"].astype("float32"), C)   # codes -> 14-vector

print(f"ADC      : max|diff| vs golden codes = {np.abs(codes - g['adc_codes']).max():.3e}")
print(f"network  : max|diff| vs golden y14   = {np.abs(y_ref - g['y_pred_14']).max():.3e}")

names = ["x", "sig_x", "y", "sig_y", "cotA", "sig_A", "cotB", "sig_B",
         "M21", "M31", "M32", "M41", "M42", "M43"]
d = np.abs(y_ref - g["y_pred_14"]).max(0)
print("\nper-slot max|diff| (diagnostic -- shows which datapath drifts first):")
for n, v in zip(names, d):
    print(f"  {n:<6}{v:.3e}")

## 3. The physics

I68 is the group's metric: half-width of the minimal interval containing 68% of residuals. Use it,
not the residual standard deviation — the std is tail-driven and has produced non-monotonic
nonsense on this problem before.

Sign accuracy is reported separately and is the number to watch in hardware: the gate reads a
*difference of two nearly equal centroids*, so reduced precision hits it long before it moves the
positions.

In [ ]:
# ---- 3. residuals, I68, sign accuracy ------------------------------------
def i68(r):
    r = np.sort(np.asarray(r)[np.isfinite(r)]); n = len(r); k = int(np.ceil(0.68 * n))
    return (r[-1] - r[0]) / 2.0 if k >= n else (r[k:] - r[:n - k]).min() / 2.0

mu, sig = decode(y_ref, ls)
yt = g["y_true_normalized"] * ls
deg = lambda c: np.degrees(np.arctan2(1.0, c))

fig, axes = plt.subplots(1, 4, figsize=(19, 3.6))
rows = []
for i, (nm, unit, ax) in enumerate([("x", "um", axes[0]), ("y", "um", axes[1]),
                                    ("alpha", "deg", axes[2]), ("beta", "deg", axes[3])]):
    if unit == "um":
        t, p, sg = yt[:, i], mu[:, i], sig[:, i]
    else:
        # sigma must be converted to degrees too, or the pull is meaningless:
        # the residual is an angle difference while sig[:, i] is still in cot units
        t, p = deg(yt[:, i]), deg(mu[:, i])
        sg = 0.5 * (np.abs(deg(mu[:, i] + sig[:, i]) - p) + np.abs(deg(mu[:, i] - sig[:, i]) - p))
    res = t - p
    ax.hist(res, bins=60, histtype="step", lw=1.6, color="tab:blue")
    ax.axvline(0, color="gray", ls="--", alpha=0.5)
    ax.set_xlabel(f"true - predicted {nm} [{unit}]")
    ax.set_title(f"{nm}:  I68 = {i68(res):.3f} {unit}")
    sa = np.nan if i < 2 else (np.sign(mu[:, i]) == np.sign(yt[:, i])).mean()
    rows.append((nm, unit, res.mean(), i68(res), (res / (sg + 1e-12)).std(), sa))
plt.tight_layout(); plt.show()

print(f"{'out':<7}{'unit':>5}{'mean':>10}{'I68':>10}{'pull std':>10}{'sign acc':>10}")
for nm, unit, m, q, pu, sa in rows:
    print(f"{nm:<7}{unit:>5}{m:10.3f}{q:10.3f}{pu:10.3f}" + ("" if np.isnan(sa) else f"{sa:10.4f}"))
print("\n(pull std ~ 1.0 means the predicted sigmas are calibrated)")

## 4. How much precision does the datapath need?

The mean path is: accumulate profiles, two barycenter divisions, two width counts, four drift
divisions, two `tanh`. This sweeps the precision of the two quantities that carry the most
information — the **centroid** and the **between-slice drift** — by rounding each to a fixed step,
then re-runs the formula.

It is a software stand-in for a fixed-point study, not a substitute for one, but it tells you where
to spend bits before you build anything.

In [ ]:
# ---- 4. precision sensitivity --------------------------------------------
# Re-implements the N=1 mean path with an explicit rounding step on the centroids and
# the drifts, so each can be degraded independently.
E, geom = C["experts"][0], C["geometry"]
xc = (np.arange(16) - 7.5) * geom["p_x"]
yc = (np.arange(16) - 7.5) * geom["p_y"]
D = g["adc_codes"].astype("float32")

def means_with_precision(step_centroid=0.0, step_drift=0.0):
    rnd = lambda v, s: v if s <= 0 else np.round(v / s) * s
    q = D.sum(-1); Px, Py = q.sum(1), q.sum(2)
    xb = rnd((Px * xc).sum(-1) / (Px.sum(-1) + 1e-6), step_centroid)
    yb = rnd((Py * yc).sum(-1) / (Py.sum(-1) + 1e-6), step_centroid) + E["dy_over_2"]
    Wx, Wy = (Px > 0).sum(-1), (Py > 0).sum(-1)
    ca = np.maximum(Wx - 1, 0) * geom["p_x"] / geom["T"]
    cb = np.maximum(Wy - 1, 0) * geom["p_y"] / geom["T"] + E["lorentz_term"]
    q0, q1 = np.maximum(D[..., 0], 0), np.maximum(D[..., 1], 0)
    cen = lambda a, c: (a * c).sum(-1) / (a.sum(-1) + 1e-6)
    Tx = rnd(np.clip(cen(q1.sum(1), xc) - cen(q0.sum(1), xc), -800, 800), step_drift)
    Ty = rnd(np.clip(cen(q1.sum(2), yc) - cen(q0.sum(2), yc), -200, 200), step_drift)
    sa = np.tanh(E["sign_k_alpha"] * (E["sign_a0_alpha"] - Tx))
    sb = np.tanh(E["sign_k_beta"] * (E["sign_a0_beta"] - Ty))
    raw = np.stack([xb, yb, ca * sa, cb * sb], -1) / ls
    return (np.asarray(E["aff_scale"]) * raw + np.asarray(E["aff_bias"])) * ls

steps = [0.0, 0.25, 0.5, 1.0, 2.0, 4.0, 8.0]          # micrometres
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for lbl, kw, style in [("centroid step", "step_centroid", "o-"), ("drift step", "step_drift", "s--")]:
    ix, isg, ibg = [], [], []
    for s in steps:
        m = means_with_precision(**{kw: s})
        ix.append(i68(yt[:, 0] - m[:, 0]))
        isg.append((np.sign(m[:, 2]) == np.sign(yt[:, 2])).mean())
        ibg.append((np.sign(m[:, 3]) == np.sign(yt[:, 3])).mean())
    axes[0].plot(steps, ix, style, label=lbl)
    axes[1].plot(steps, isg, style, label=lbl)
    axes[2].plot(steps, ibg, style, label=lbl)
for ax, t, yl in [(axes[0], "position", "I68(x) [um]"),
                  (axes[1], "sign alpha", "accuracy"), (axes[2], "sign beta", "accuracy")]:
    ax.set_xlabel("rounding step [um]"); ax.set_ylabel(yl); ax.set_title(t); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

print("read it as: how coarse can this quantity get before the output moves?")
print("pitches are 50 um (x) and 12.5 um (y), so a 1 um step is ~1/50 and ~1/12 of a pixel")

## 5. Optional — rebuild the Keras model

Only if you want to re-derive the constants or push new data through the trained graph. Needs
**TF 2.15 + qkeras**; on the Purdue AF that is the `SmartPixels` kernel
(`/work/users/kuang14/smart_pixels/bin/python`). Everything above works without it.

In [ ]:
# ---- 5. rebuild from weights (needs TF) ----------------------------------
import sys
sys.path.insert(0, "/depot/cms/private/users/kuang14/Smart_Pixel/smart-pixels-digitization/two_bit_optimization_helpers")
from symbolic.moe import load_arm

RUN = f"/depot/cms/private/users/kuang14/Smart_Pixel/{ARM}"
model, core, summary = load_arm(RUN)
y_keras = model(g["charge_analog_e"][:256], training=False).numpy()   # NOTE: takes ANALOG in
print("keras vs numpy reference:", np.abs(y_keras - y_ref[:256]).max())
print("keras vs golden         :", np.abs(y_keras - g["y_pred_14"][:256]).max())
print("\nthe Keras model digitizes internally, so it takes analog charge;")
print("symbolic_ref.forward() takes ADC codes, matching the hardware boundary.")